In [5]:
!pip install numpy pandas matplotlib scikit-fuzzy


In [6]:
# heart_disease_fuzzy.py - Manual Mamdani Fuzzy System (no skfuzzy ControlSystem)
import numpy as np
import skfuzzy as fuzz

# Universes (same as article)
chest_pain_univ = np.arange(0, 8, 1)
hba1c_univ = np.arange(3, 15, 0.1)
hdl_univ = np.arange(10, 81, 1)
ldl_univ = np.arange(40, 201, 1)
heart_rate_univ = np.arange(40, 161, 1)
age_univ = np.arange(20, 121, 1)
blood_pressure_univ = np.arange(80, 241, 1)
risk_univ = np.arange(0, 11, 0.1)

# Membership functions dicts
mf_chest_pain = {
    'no_pain': fuzz.trimf(chest_pain_univ, [0, 1, 2]),
    'non_anginal': fuzz.trimf(chest_pain_univ, [2, 3, 4]),
    'atypical': fuzz.trimf(chest_pain_univ, [4, 5, 6]),
    'typical': fuzz.trimf(chest_pain_univ, [6, 7, 8])
}

mf_hba1c = {
    'very_healthy': fuzz.trimf(hba1c_univ, [3, 5, 7]),
    'healthy': fuzz.trimf(hba1c_univ, [6.5, 7.75, 9]),
    'high': fuzz.trimf(hba1c_univ, [8.5, 11.25, 14])
}

mf_hdl = {
    'low': fuzz.trimf(hdl_univ, [10, 30, 50]),
    'healthy': fuzz.trimf(hdl_univ, [40, 60, 80])
}

mf_ldl = {
    'very_healthy': fuzz.trimf(ldl_univ, [40, 60, 80]),
    'healthy': fuzz.trimf(ldl_univ, [70, 90, 110]),
    'high': fuzz.trimf(ldl_univ, [100, 120, 140]),
    'very_high': fuzz.trimf(ldl_univ, [130, 150, 170]),
    'extra_high': fuzz.trimf(ldl_univ, [160, 180, 200])
}

mf_heart_rate = {
    'very_healthy': fuzz.trimf(heart_rate_univ, [40, 55, 70]),
    'healthy': fuzz.trimf(heart_rate_univ, [60, 80, 100]),
    'high': fuzz.trimf(heart_rate_univ, [90, 125, 160])
}

mf_age = {
    'young': fuzz.trimf(age_univ, [20, 32.5, 45]),
    'mid': fuzz.trimf(age_univ, [40, 52.5, 65]),
    'old': fuzz.trimf(age_univ, [60, 72.5, 85]),
    'very_old': fuzz.trimf(age_univ, [80, 100, 120])
}

mf_blood_pressure = {
    'normal': fuzz.trimf(blood_pressure_univ, [80, 110, 140]),
    'high': fuzz.trimf(blood_pressure_univ, [120, 160, 200]),
    'very_high': fuzz.trimf(blood_pressure_univ, [180, 210, 240])
}

mf_risk = {
    'healthy': fuzz.trimf(risk_univ, [0, 2, 4]),
    'low': fuzz.trimf(risk_univ, [2, 4, 6]),
    'medium': fuzz.trimf(risk_univ, [4, 6, 8]),
    'high': fuzz.trimf(risk_univ, [6, 8, 10])
}

# All linguistic labels
labels = {
    'chest_pain': list(mf_chest_pain.keys()),
    'hba1c': list(mf_hba1c.keys()),
    'hdl': list(mf_hdl.keys()),
    'ldl': list(mf_ldl.keys()),
    'heart_rate': list(mf_heart_rate.keys()),
    'age': list(mf_age.keys()),
    'blood_pressure': list(mf_blood_pressure.keys())
}

# Manual predict function
def predict_risk(cp, hba, hd, ld, hr, ag, bp):
    inputs = {'chest_pain': cp, 'hba1c': hba, 'hdl': hd, 'ldl': ld,
              'heart_rate': hr, 'age': ag, 'blood_pressure': bp}

    # Aggregated output membership
    aggregated = np.zeros_like(risk_univ)

    # Loop over all possible combinations (4320)
    import itertools
    for combo in itertools.product(*labels.values()):
        # Firing strength (MIN of memberships)
        firing = min(
            fuzz.interp_membership(chest_pain_univ, mf_chest_pain[combo[0]], inputs['chest_pain']),
            fuzz.interp_membership(hba1c_univ, mf_hba1c[combo[1]], inputs['hba1c']),
            fuzz.interp_membership(hdl_univ, mf_hdl[combo[2]], inputs['hdl']),
            fuzz.interp_membership(ldl_univ, mf_ldl[combo[3]], inputs['ldl']),
            fuzz.interp_membership(heart_rate_univ, mf_heart_rate[combo[4]], inputs['heart_rate']),
            fuzz.interp_membership(age_univ, mf_age[combo[5]], inputs['age']),
            fuzz.interp_membership(blood_pressure_univ, mf_blood_pressure[combo[6]], inputs['blood_pressure'])
        )

        if firing > 0:
            # Determine output label based on bad factors (same logic as before)
            bad_count = 0
            if combo[0] in ['atypical', 'typical']: bad_count += 2
            if combo[1] == 'high': bad_count += 2
            if combo[2] == 'low': bad_count += 2
            if combo[3] in ['very_high', 'extra_high']: bad_count += 3
            elif combo[3] == 'high': bad_count += 2
            if combo[4] == 'high': bad_count += 1
            if combo[5] in ['old', 'very_old']: bad_count += 2
            if combo[6] == 'very_high': bad_count += 3
            elif combo[6] == 'high': bad_count += 2

            if bad_count <= 3: out_label = 'healthy'
            elif bad_count <= 7: out_label = 'low'
            elif bad_count <= 11: out_label = 'medium'
            else: out_label = 'high'

            # Clip output mf
            clipped = np.fmin(firing, mf_risk[out_label])
            aggregated = np.fmax(aggregated, clipped)

    # Defuzzification (centroid - simple and close to article's average)
    if np.sum(aggregated) == 0:
        return 5.0  # fallback
    risk_value = fuzz.defuzz(risk_univ, aggregated, 'centroid')
    return risk_value

In [ ]:
# app.py - Simple version: Manual Input + Dataset Upload (NO CHARTS)
from flask import Flask, request, render_template, flash
import pandas as pd
import numpy as np
import os
from werkzeug.utils import secure_filename
from heart_disease_fuzzy import predict_risk

app = Flask(__name__)
app.secret_key = 'super_secret_key'
app.config['UPLOAD_FOLDER'] = 'uploads'
app.config['ALLOWED_EXTENSIONS'] = {'csv'}

# فقط پوشه آپلود رو بساز (پوشه charts لازم نیست)
os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)

def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in app.config['ALLOWED_EXTENSIONS']

def process_dataset(df):
    if df.empty:
        return None, "<p>The dataset is empty.</p>"

    # رفع مشکل دیتاست Cleveland (جایگزینی ? با NaN و تبدیل به عدد)
    df = df.replace('?', np.nan)
    numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'fbs', 'cp']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # حذف ردیف‌های نامعتبر
    df = df.dropna(subset=['age', 'cp', 'trestbps', 'chol', 'thalach', 'fbs'])

    # مپینگ ورودی‌ها بر اساس مقاله
    df['chest_pain_mapped'] = df['cp'].map({1: 7, 2: 5, 3: 3, 4: 1}).fillna(1)
    df['hba1c'] = np.where(df['fbs'] == 1, 9.0, 5.5)
    df['hdl'] = df['chol'] * 0.25
    df['ldl'] = df['chol'] * 0.45
    df['heart_rate'] = df['thalach']
    df['age_mapped'] = df['age']
    df['bp_mapped'] = df['trestbps']

    # پیش‌بینی ریسک
    risks = []
    levels = []
    for _, row in df.iterrows():
        try:
            risk_val = predict_risk(
                float(row['chest_pain_mapped']),
                float(row['hba1c']),
                float(row['hdl']),
                float(row['ldl']),
                float(row['heart_rate']),
                float(row['age_mapped']),
                float(row['bp_mapped'])
            )
            risk_val = round(risk_val, 2)
            risks.append(risk_val)
            level = ("Healthy" if risk_val < 4 else
                     "Low Risk" if risk_val < 6 else
                     "Medium Risk" if risk_val < 8 else "High Risk")
            levels.append(level)
        except:
            risks.append(None)
            levels.append("Error")

    df['predicted_risk'] = risks
    df['risk_level'] = levels

    # آمار خلاصه
    valid_count = df['predicted_risk'].notna().sum()
    stats = {
        'total': len(df),
        'valid_predictions': valid_count,
        'avg_risk': round(df['predicted_risk'].mean(), 2) if valid_count > 0 else 0,
        'high_risk_count': (df['risk_level'] == 'High Risk').sum(),
        'high_risk_percent': round(((df['risk_level'] == 'High Risk').sum() / len(df)) * 100, 1)
    }

    # جدول نمونه
    display_cols = ['age', 'cp', 'trestbps', 'chol', 'thalach', 'predicted_risk', 'risk_level']
    summary_html = pd.concat([df.head(10), df.tail(10)])[display_cols].to_html(
        classes='table table-striped', index=False, na_rep='N/A')

    return stats, summary_html

@app.route('/', methods=['GET', 'POST'])
def index():
    manual_result = None
    manual_score = None
    manual_level = None
    uploaded_stats = None
    uploaded_table = "<p>No dataset uploaded yet.</p>"

    # ورودی دستی
    if request.method == 'POST' and 'chest_pain' in request.form:
        try:
            inputs = {
                'cp': float(request.form['chest_pain']),
                'hba': float(request.form['hba1c']),
                'hd': float(request.form['hdl']),
                'ld': float(request.form['ldl']),
                'hr': float(request.form['heart_rate']),
                'ag': float(request.form['age']),
                'bp': float(request.form['blood_pressure'])
            }
            risk_value = round(predict_risk(**inputs), 2)
            level = ("Healthy" if risk_value < 4 else
                     "Low Risk" if risk_value < 6 else
                     "Medium Risk" if risk_value < 8 else "High Risk")
            manual_result = "success"
            manual_score = risk_value
            manual_level = level
        except:
            manual_result = "error"
            manual_level = "Please enter valid numeric values."

    # آپلود دیتاست
    if request.method == 'POST' and 'file' in request.files:
        file = request.files['file']
        if file.filename == '':
            flash('No file selected.', 'danger')
        elif file and allowed_file(file.filename):
            filename = secure_filename(file.filename)
            filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
            file.save(filepath)
            try:
                # تشخیص خودکار فرمت Cleveland
                if 'cleveland' in filename.lower():
                    df = pd.read_csv(filepath, header=None, na_values='?')
                    df.columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
                                  'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']
                else:
                    df = pd.read_csv(filepath, na_values='?')

                uploaded_stats, uploaded_table = process_dataset(df)
                flash('Dataset analyzed successfully!', 'success')
            except Exception as e:
                flash(f'Error processing file: {str(e)}', 'danger')
        else:
            flash('Only CSV files are allowed.', 'danger')

    return render_template('index.html',
                           manual_result=manual_result,
                           manual_score=manual_score,
                           manual_level=manual_level,
                           uploaded_stats=uploaded_stats,
                           uploaded_table=uploaded_table)

if __name__ == '__main__':
    app.run(host='127.0.0.1', port=5000, debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)
